# Setup

Load the static timetable tables written by the schedule refresh, plus the realtime trip updates to test the join against.
Everything is worked out inline so the logic is visible here before it lives in `src/`.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
conn = sqlite3.connect(PROJECT_ROOT / "data/traffic.db")

routes_df = pd.read_sql("SELECT * FROM routes", conn)
trips_df = pd.read_sql("SELECT * FROM trips", conn)
stops_df = pd.read_sql("SELECT * FROM stops", conn)
stop_times_df = pd.read_sql("SELECT * FROM stop_times", conn)
calendar_df = pd.read_sql("SELECT * FROM calendar", conn)
calendar_dates_df = pd.read_sql("SELECT * FROM calendar_dates", conn)
trip_updates_df = pd.read_sql("SELECT * FROM trip_updates WHERE arrival_time > 0", conn)

for name, df in [("routes", routes_df), ("trips", trips_df), ("stops", stops_df), ("stop_times", stop_times_df), ("calendar", calendar_df), ("calendar_dates", calendar_dates_df)]:
    print(f"{name:15} {df.shape}")

routes          (1, 9)
trips           (1352, 11)
stops           (42, 10)
stop_times      (28348, 11)
calendar        (4, 10)
calendar_dates  (1, 3)


# What the schedule covers

One route, and a calendar that says which service runs on which days and for how long this timetable is valid.

In [2]:
routes_df[["route_id", "agency_id", "route_short_name", "route_long_name"]]

,route_id,agency_id,route_short_name,route_long_name
0,3-M1-sj2-1,SMNW,M1,M1 Metro North West and Bankstown Line


In [3]:
print(calendar_df.to_string(index=False))
print()
calendar_dates_df

 service_id monday tuesday wednesday thursday friday saturday sunday start_date end_date
   TA+az+11      0       0         0        0      0        0      1   20260910 20261209
  TA+R59+11      0       0         0        0      1        0      0   20260910 20261209
 TA+TT21+11      0       0         0        0      0        1      0   20260910 20261209
TA+v9500+11      1       1         1        1      0        0      0   20260910 20261209



,service_id,date,exception_type
0,TA+az+11,20260913,2


# Stops

Every stop is a platform, so a station appears once per direction.

In [4]:
print(len(stops_df), "stops across", stops_df["parent_station"].nunique(), "stations")
stops_df[["stop_id", "stop_name", "platform_code", "stop_lat", "stop_lon"]].head(10)

42 stops across 21 stations


,stop_id,stop_name,platform_code,stop_lat,stop_lon
0,2113341,"Macquarie Park Station, Platform 1",1,-33.78498208,151.12833711
1,2113342,"Macquarie Park Station, Platform 2",2,-33.78506593,151.12825481
2,2113351,"Macquarie University Station, Platform 1",1,-33.77759575,151.11831444
3,2113352,"Macquarie University Station, Platform 2",2,-33.77765225,151.11824162
4,2153402,"Bella Vista Station, Platform 1",1,-33.73069803,150.94412269
5,2153403,"Bella Vista Station, Platform 2",2,-33.73071878,150.94403737
6,2153404,"Norwest Station, Platform 1",1,-33.73428249,150.96385134
7,2153405,"Norwest Station, Platform 2",2,-33.73433440,150.96391844
8,2154262,"Castle Hill Station, Platform 1",1,-33.73146885,151.00768022
9,2154263,"Castle Hill Station, Platform 2",2,-33.73154912,151.00770552


# Trip patterns

How many stops a trip makes, and how trips split by direction and service.

In [5]:
stops_per_trip = stop_times_df.groupby("trip_id").size()
print(stops_per_trip.value_counts().sort_index().rename("trips").to_string())
print()
trips_df.groupby(["service_id", "direction_id"]).size().unstack(fill_value=0)

9        3
13       1
21    1348



direction_id,0,1
service_id,,
TA+R59+11,223,223
TA+TT21+11,124,125
TA+az+11,109,114
TA+v9500+11,217,217


# Scheduled headway, and a trap

The gap between one train and the next at the same platform. The stop_times table holds a copy of the timetable for every day type, so measuring gaps across all of them together produces zeros.

In [6]:
# GTFS times can pass 24:00:00 for services after midnight, so they are kept as seconds rather than parsed as clock times.
parts = stop_times_df["arrival_time"].str.split(":", expand=True).astype(int)
stop_times_df["sched_secs"] = parts[0] * 3600 + parts[1] * 60 + parts[2]
sched_df = stop_times_df.merge(trips_df[["trip_id", "service_id", "direction_id"]], on="trip_id")

naive = sched_df.sort_values(["stop_id", "direction_id", "sched_secs"]).groupby(["stop_id", "direction_id"])["sched_secs"].diff()
within = sched_df.sort_values(["service_id", "stop_id", "direction_id", "sched_secs"]).groupby(["service_id", "stop_id", "direction_id"])["sched_secs"].diff()

print(f"across all day types: median {naive.median():.0f} s, zero gaps {(naive == 0).sum()}")
print(f"within one day type : median {within.median():.0f} s, zero gaps {(within == 0).sum()}")

across all day types: median 0 s, zero gaps 16357
within one day type : median 300 s, zero gaps 0


In [7]:
sched_df["headway_secs"] = within
sched_df.groupby("service_id")["headway_secs"].describe()[["count", "min", "50%", "max"]]

,count,min,50%,max
service_id,,,,
TA+R59+11,9320.0,216.0,254.0,28800.0
TA+TT21+11,5187.0,580.0,600.0,620.0
TA+az+11,4597.0,580.0,600.0,620.0
TA+v9500+11,9068.0,216.0,252.0,28800.0


# Do realtime ids match static ids?

The realtime feed and the static schedule both carry trip, route and stop ids. Check which ones actually line up.

In [8]:
rt_trips = set(trip_updates_df["trip_id"])
st_trips = set(trips_df["trip_id"])
print(f"trip_id : {len(rt_trips)} realtime, {len(st_trips)} static, {len(rt_trips & st_trips)} shared")
print(f"route_id: realtime {trip_updates_df['route_id'].unique().tolist()} vs static {routes_df['route_id'].tolist()}")
rt_stops = set(trip_updates_df["stop_id"].astype(str))
st_stops = set(stops_df["stop_id"].astype(str))
print(f"stop_id : {len(rt_stops)} realtime, {len(st_stops)} static, {len(rt_stops & st_stops)} shared")
print()
print("realtime trip ids:", sorted(rt_trips)[:3])
print("static trip ids  :", sorted(st_trips)[:3])

trip_id : 48 realtime, 1352 static, 0 shared
route_id: realtime ['SMNW_M1'] vs static ['3-M1-sj2-1']
stop_id : 42 realtime, 42 static, 42 shared

realtime trip ids: ['0241-001-101-016:1000', '0241-001-101-017:1000', '0241-001-101-018:1000']
static trip ids  : ['0251-002-101-002:1000', '0251-002-101-003:1000', '0251-002-101-004:1000']


# The part of the trip id that survives

The first two segments change with every timetable revision. The run and trip number after them identifies the physical service and does not.

In [9]:
RUN_KEY = r"^\d+-\d+-(\d+-\d+):"
rt_keys = set(trip_updates_df["trip_id"].str.extract(RUN_KEY)[0].dropna())
trips_df["run_key"] = trips_df["trip_id"].str.extract(RUN_KEY)[0]
st_keys = set(trips_df["run_key"].dropna())
print(f"run keys: {len(rt_keys)} realtime, {len(st_keys)} static, {len(rt_keys & st_keys)} shared")
print("unmatched realtime run keys:", sorted(rt_keys - st_keys))
print()
print("static trips per run key:", trips_df.groupby("run_key").size().value_counts().sort_index().to_dict())
print("run keys whose variants disagree on direction:", (trips_df.groupby("run_key")["direction_id"].nunique() > 1).sum())

run keys: 48 realtime, 480 static, 45 shared
unmatched realtime run keys: ['113-011', '113-012', '113-013']

static trips per run key: {1: 22, 2: 243, 3: 16, 4: 199}
run keys whose variants disagree on direction: 0


In [10]:
example = sorted(rt_keys & st_keys)[0]
trips_df[trips_df["run_key"] == example][["trip_id", "service_id", "direction_id", "trip_headsign"]]

,trip_id,service_id,direction_id,trip_headsign
14,0251-002-101-016:1000,TA+v9500+11,1,Sydenham
448,0252-001-101-016:1000,TA+R59+11,1,Sydenham
894,0253-002-101-016:1000,TA+TT21+11,1,Sydenham
1143,0254-001-101-016:1000,TA+az+11,1,Sydenham


# Why scheduled times do not line up either

GTFS-Realtime defines delay as predicted minus scheduled, so arrival_time minus arrival_delay recovers the scheduled arrival the feed was working from. Compare that against the static schedule at the same stop.

In [11]:
rt = trip_updates_df.copy()
rt["stop_id"] = rt["stop_id"].astype(str)
sched_local = pd.to_datetime(rt["arrival_time"] - rt["arrival_delay"], unit="s", utc=True).dt.tz_convert("Australia/Sydney")
midnight = pd.to_datetime(rt["start_date"], format="%Y%m%d").dt.tz_localize("Australia/Sydney")
rt["sched_secs"] = (sched_local - midnight).dt.total_seconds().astype(int)

rt_times = rt[["stop_id", "sched_secs"]].drop_duplicates()
static_times = sched_df[["stop_id", "sched_secs"]].drop_duplicates().sort_values("sched_secs")
exact = rt_times.merge(static_times, on=["stop_id", "sched_secs"], how="inner")
nearest = pd.merge_asof(rt_times.sort_values("sched_secs"), static_times.rename(columns={"sched_secs": "static_secs"}), left_on="sched_secs", right_on="static_secs", by="stop_id", direction="nearest")
gap = (nearest["sched_secs"] - nearest["static_secs"]).abs()

print(f"realtime scheduled times: {len(rt_times)}, exactly present in static: {len(exact)}")
print(f"gap to nearest static time at the same stop: median {gap.median():.0f} s, 75th percentile {gap.quantile(0.75):.0f} s")

realtime scheduled times: 961, exactly present in static: 1
gap to nearest static time at the same stop: median 63 s, 75th percentile 123 s


# Resolving which variant ran on a given day

Each run key has one static trip per day-type service, so the calendar picks the right one from the service day.

In [12]:
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

for date in ["20260815", "20260910", "20260912", "20260913"]:
    weekday = WEEKDAYS[pd.Timestamp(date).dayofweek]
    in_range = (calendar_df["start_date"] <= date) & (calendar_df["end_date"] >= date)
    active = set(calendar_df.loc[in_range & (calendar_df[weekday] == "1"), "service_id"])
    exceptions = calendar_dates_df[calendar_dates_df["date"] == date]
    active |= set(exceptions.loc[exceptions["exception_type"] == "1", "service_id"])
    active -= set(exceptions.loc[exceptions["exception_type"] == "2", "service_id"])
    print(f"{date} ({weekday:9}): {sorted(active)}")

20260815 (saturday ): []
20260910 (thursday ): ['TA+v9500+11']
20260912 (saturday ): ['TA+TT21+11']
20260913 (sunday   ): []


# Findings

- The realtime and static feeds share stop ids completely, but not trip ids or route ids. The join key is the run and trip number inside the trip id, which matches 45 of the 48 realtime trips.
- Static holds one copy of each trip per day type, four in all, with the same run key. The calendar picks the copy for a given service day. The collected realtime data predates this timetable, so for it the calendar returns nothing and the join falls back to the first copy.
- Scheduled times from this static download do not match the times the realtime feed was measured against, because the timetable was revised in between. The gap is about a minute. Static can supply stop names, direction, position along the trip and headway, but delay must come from the feed itself.
- Headway has to be measured within one day type, or the duplicate copies produce zero gaps. Done that way, weekday headway is about four minutes and weekend about ten.
- Trips run 9, 13 or 21 stops, and every variant of a run key agrees on direction.

# Next step

The run key match, the calendar resolution and the per-service headway are the pieces that need to become the static join in the pipeline, with the match method recorded on every row so downstream work knows how much to trust it.